<a href="https://colab.research.google.com/github/janithcyapa/Wearable-Biomechanics/blob/main/Dataset_Manager.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **🏋🏽 Wearable Biomechanics - Dataset Manager**
> MARC, UOP


This notebook serves as the central command interface for the **Wearable Biomechanics** project. It bridges the gap between raw Google Drive storage and Google Colab by using the **[OpenDatasetStore](https://github.com/janithcyapa/OpenDatasetStore)** utility.

The system maintains an automated, **JSON-indexed manifest**, ensuring that every subject and experiment is strictly organized with unique IDs and searchable metadata.

* **🛠️ Framework:** [OpenDatasetStore Library](https://github.com/janithcyapa/OpenDatasetStore)
* **💻 Project Code:** [Wearable-Biomechanics Repository](https://github.com/janithcyapa/Wearable-Biomechanics/)
* **📂 Data Storage:** [Project Google Drive Folder](https://drive.google.com/drive/folders/1QERF02T4pvubAEjnv6f1VrCOWNYxHv54?usp=drive_link)


## **🛠️ Setup OpenDatasetStore**

In [ ]:

from google.colab import drive
import pandas as pd
import os

# Mount Google Drive
drive.mount('/content/drive', force_remount=True)

# Download the latest store manager
!wget -q -O open_dataset_store.py https://raw.githubusercontent.com/janithcyapa/OpenDatasetStore/refs/heads/main/open_dataset_store.py

from open_dataset_store import OpenDatasetStore

# @markdown ### Store Configuration
dataset_path = "/content/drive/MyDrive/Datasets/Wearable-Biomechanics" # @param {type:"string"}

store = OpenDatasetStore(
    base_dir=dataset_path,
    entity_id_format=f"sub_{{num:04d}}",
    entry_id_format="{type}_{num:04d}",
    raw_filename_format="{entry_id}_{entity_id}_{ts}.csv",
    processed_filename_format="{entry_id}_{tag}.parquet"
)

# **👤 People Management (Subject)**

In [ ]:
# @title 📝 Register New Subject { display-mode: "form" }
# @markdown Enter the details of the test subject to generate a unique ID.

name = "Hivindu Kirulapana" # @param {type:"string"}
age = 22 # @param {type:"number"}
weight_kg = 72 # @param {type:"number"}
height_m = 1.74 # @param {type:"number"}
experience_years = 1 # @param {type:"integer"}
notes = "Participation in Other Sports - Rugby" # @param {type:"string"}
achievement = ""# @param {type:"string"}
best_weight = ""# @param {type:"string"}

# Logic to create entity
sub_id = store.create_entity(
    entity_type="subjects",
    name=name,
    age=age,
    weight=weight_kg,
    height=height_m,
    experience=experience_years,
    achievement=achievement,
    best_weight=best_weight,
    notes=notes
)

print(f"✅ Created Subject ID: {sub_id}")

In [ ]:
# @title 📋 Participant Registry { display-mode: "form" }
# @markdown View and filter the registered test subjects.

from google.colab import data_table

# Fetch subject data
df_subjects = store.list_entities("subjects")

if not df_subjects.empty:
    # 1. Clean up column names for display (e.g., replace underscores with spaces)
    df_subjects.columns = [c.replace('_', ' ').title() for c in df_subjects.columns]

    # 2. Display using Colab's Interactive Data Table
    print(f"Total Subjects Registered: {len(df_subjects)}")
    display(data_table.DataTable(
        df_subjects,
        include_index=False,
        num_rows_per_page=10,
        max_columns=20
    ))
else:
    print("⚠️ No participants found in the registry.")

# **📂 Experiment Upload & Data Exploration**

In [ ]:
# @title 🚀 Upload New Experiment { display-mode: "form" }
# @markdown Select a subject and upload their raw IMU data.

target_subject_id = "subj_0001" # @param {type:"string"}
activity_type = "Clean & Jerk" # @param {type:"string"}
weight_used = 20 # @param {type:"number"}

exp_id = store.create_entry_interactive(
    entry_type="experiments",
    entity_id=target_subject_id,
    description=activity_type,
    weight_kg=weight_used
)

# @markdown ---
# @markdown ### 📈 Auto-Visualization
# @markdown After upload, the IMU data will be plotted below for quality check.

if exp_id:
    raw_df = store.get_entry_raw_data("experiments", exp_id)

    import plotly.graph_objects as go
    from plotly.subplots import make_subplots

    # Dynamically find Accelerometer columns (assuming AX, AY, AZ naming)
    accel_cols = [c for c in raw_df.columns if '_AX' in c or '_AY' in c or '_AZ' in c][:3]

    fig = go.Figure()
    for col in accel_cols:
        fig.add_trace(go.Scatter(y=raw_df[col], name=col, mode='lines'))

    fig.update_layout(
        title=f"Initial Data Check: {exp_id}",
        xaxis_title="Samples",
        yaxis_title="Acceleration ($m/s^2$)",
        template="plotly_dark",
        height=400
    )
    fig.show()

In [ ]:
# @title 📎 Attachment Manager { display-mode: "form" }
# @markdown Upload related files (videos, high-speed clips, or archives) to an experiment.

experiment_to_attach = "experiments_0001" # @param {type:"string"}
file_label = "front_video" # @param ["front_video", "side_video", "raw_zip", "other"]

rel_path = store.upload_related_file_interactive(
    entry_type="experiments",
    entry_id=experiment_to_attach,
    label=file_label
)

In [4]:
# @title 🧪 Experiment Log { display-mode: "form" }
# @markdown Browse all uploaded biomechanical data sessions.

# Fetch experiment entries
df_experiments = store.list_entries("experiments")

if not df_experiments.empty:
    # Organize columns so the most important info is first
    cols_priority = ["entry_id", "entity_id", "timestamp", "description"]
    other_cols = [c for c in df_experiments.columns if c not in cols_priority]
    df_experiments = df_experiments[cols_priority + other_cols]

    # Format the table for a cleaner look
    df_experiments.columns = [c.replace('_', ' ').title() for c in df_experiments.columns]

    display(data_table.DataTable(
        df_experiments,
        include_index=False,
        num_rows_per_page=10
    ))
else:
    print("⚠️ No experiments recorded yet.")

No entries for type 'experiments'.
⚠️ No experiments recorded yet.


# **📊  Advanced IMU Analysis Plot**

In [ ]:
# @title 🔍 Detailed Signal Analysis { display-mode: "form" }
# @markdown Compare different IMU nodes on the body.

selected_exp = "experiments_0001" # @param {type:"string"}
sensor_node_A = "OrbiX-Node-1" # @param {type:"string"}
sensor_node_B = "OrbiX-Node-9" # @param {type:"string"}

df_analysis = store.get_entry_raw_data("experiments", selected_exp)

import plotly.express as px

# Filter for specific nodes
plot_cols = [c for c in df_analysis.columns if sensor_node_A in c or sensor_node_B in c]
plot_cols = [c for c in plot_cols if "IMU1_AX" in c] # Just showing X-axis for comparison

fig = px.line(df_analysis, y=plot_cols,
              title=f"Comparison: {sensor_node_A} vs {sensor_node_B}",
              labels={"value": "G-Force", "index": "Time (Samples)"},
              template="simple_white")

fig.show()